## Cursores

Calcular la cantidad total acumulada para cada cliente y mes desde la vista ClientesOrdenes

In [1]:
SET NOCOUNT ON;

DECLARE @Resultado AS TABLE (
IDCliente           char(5), 
mesorden            DATE, 
cantidad            INT,
cantidadacumulada   INT,
PRIMARY KEY(IDCliente, mesorden)
);

DECLARE
@clienteid          AS char(5), 
@antclienteid       AS char(5), 
@mesorden           AS DATE,
@cantidad           AS INT,
@cantidadacumulada  AS INT;

DECLARE C CURSOR FAST_FORWARD 
/* solo lectura, solo avanza */ 
FOR SELECT IDCliente, MesOrden, cantidad
FROM ClientesOrdenes
ORDER BY IDCliente, MesOrden; 

OPEN C;
FETCH NEXT FROM C INTO @clienteid, @mesorden, @cantidad; 
SELECT @antclienteid = @clienteid, @cantidadacumulada = 0;
WHILE @@FETCH_STATUS = 0 
    BEGIN
        IF @clienteid <> @antclienteid
            SELECT @antclienteid = @clienteid, @cantidadacumulada = 0; 
        SET @cantidadacumulada = @cantidadacumulada + @cantidad;
        INSERT INTO @Resultado 
        VALUES(@clienteid, @mesorden, @cantidad, @cantidadacumulada);
        FETCH NEXT FROM C INTO @clienteid, @mesorden, @cantidad; 
    END;
CLOSE C; 
DEALLOCATE C; 
SELECT IDCliente, CONVERT(VARCHAR(7), mesorden, 121) AS mesorden, cantidad,
cantidadacumulada 
FROM @Resultado
ORDER BY IDCliente, mesorden;

Commands completed successfully.

IDCliente | mesorden | cantidad | cantidadacumulada
----------+----------+----------+------------------
ALFKI     | 2020-08  | 38       | 38               
ALFKI     | 2020-10  | 41       | 79               
ALFKI     | 2021-01  | 17       | 96               
ALFKI     | 2021-03  | 18       | 114              
ALFKI     | 2021-04  | 60       | 174              
ANATR     | 2019-09  | 6        | 6                
ANATR     | 2020-08  | 18       | 24               
ANATR     | 2020-11  | 10       | 34               
ANATR     | 2021-03  | 29       | 63               
ANTON     | 2019-11  | 24       | 24               
ANTON     | 2020-04  | 30       | 54               
ANTON     | 2020-05  | 80       | 134              
ANTON     | 2020-06  | 83       | 217              
ANTON     | 2020-09  | 102      | 319              
ANTON     | 2021-01  | 40       | 359              
AROUT     | 2019-11  | 50       | 50               
AROUT     | 2019-12  | 55     

### Con Funciones de Ventana

In [2]:
SELECT IDCliente, MesOrden, cantidad, 
SUM(cantidad) OVER(PARTITION BY IDCliente
ORDER BY MesOrden
ROWS UNBOUNDED PRECEDING) AS cantidadacumulada
FROM ClientesOrdenes
ORDER BY IDCliente, MesOrden;

Commands completed successfully.

IDCliente | MesOrden   | cantidad | cantidadacumulada
----------+------------+----------+------------------
ALFKI     | 2020-08-01 | 38       | 38               
ALFKI     | 2020-10-01 | 41       | 79               
ALFKI     | 2021-01-01 | 17       | 96               
ALFKI     | 2021-03-01 | 18       | 114              
ALFKI     | 2021-04-01 | 60       | 174              
ANATR     | 2019-09-01 | 6        | 6                
ANATR     | 2020-08-01 | 18       | 24               
ANATR     | 2020-11-01 | 10       | 34               
ANATR     | 2021-03-01 | 29       | 63               
ANTON     | 2019-11-01 | 24       | 24               
ANTON     | 2020-04-01 | 30       | 54               
ANTON     | 2020-05-01 | 80       | 134              
ANTON     | 2020-06-01 | 83       | 217              
ANTON     | 2020-09-01 | 102      | 319              
ANTON     | 2021-01-01 | 40       | 359              
AROUT     | 2019-11-01 | 50       | 50          

## SQL Dinámico

In [3]:
DECLARE @sql AS VARCHAR(100);
SET @sql = 'PRINT ''Este mensaje fue impreso por un lote de SQL dinamico.'';'; 
EXEC(@sql);

Este mensaje fue impreso por un lote de SQL dinamico.

Total execution time: 00:00:00.017

In [4]:
DECLARE @sql AS NVARCHAR(100);

SET @sql = N'SELECT IDPedido, IDCliente, IDEmpleado, FechaPedido FROM Pedidos WHERE IDPedido = @ordenid;';

EXEC sp_executesql @stmt = @sql, @params = N'@ordenid AS INT', @ordenid = 10248;

Commands completed successfully.

IDPedido | IDCliente | IDEmpleado | FechaPedido            
---------+-----------+------------+------------------------
10248    | VINET     | 5          | 2019-07-04 16:58:00.000
(1 row)

Total execution time: 00:00:00.013

In [5]:
DECLARE @IntVariable INT;
DECLARE @SQLString NVARCHAR(500);
DECLARE @ParmDefinition NVARCHAR(500);

/* Construimos la cadena SQL una sola vez.*/
SET @SQLString = 
    N'SELECT IDEmpleado, Apellido, Puesto, FechaAlta
    FROM Empleados
    WHERE jefeid = @IDJefe';
SET @ParmDefinition = N'@IDJefe int';
/* Ejecutamos con el primer valor. */
SET @IntVariable = 2;
EXECUTE sp_executesql @SQLString, @ParmDefinition,
                        @IDJefe = @IntVariable;
/* Ejecutamos la misma cadena con un segundo valor. */
SET @IntVariable = 5;
EXECUTE sp_executesql @SQLString, @ParmDefinition,
                        @IDJefe = @IntVariable;

Commands completed successfully.

IDEmpleado | Apellido  | Puesto                         | FechaAlta 
-----------+-----------+--------------------------------+-----------
1          | Davolio   | Representante de Ventas        | 2015-05-01
3          | Leverling | Representante de Ventas        | 2015-04-01
4          | Peacock   | Representante de Ventas        | 2016-05-03
5          | Buchanan  | Gerente de Ventas              | 2016-10-17
8          | Callahan  | Coordinador de Ventas Internas | 2017-03-05
(5 rows)

IDEmpleado | Apellido  | Puesto                  | FechaAlta 
-----------+-----------+-------------------------+-----------
6          | Suyama    | Representante de Ventas | 2016-10-17
7          | King      | Representante de Ventas | 2017-01-02
9          | Dodsworth | Representante de Ventas | 2017-11-15
(3 rows)

Total execution time: 00:00:00.038

## Rutinas
### Función Escalar

In [6]:
USE WideWorldImporters
SELECT CustomerID, Website.CalculateCustomerPrice(CustomerID, 1, '20160101') AS PrecProd1
FROM Sales.customers
ORDER BY CustomerID;
SELECT StockItemID, Website.CalculateCustomerPrice(1, StockItemID, '20160101') AS PrecCli1
FROM Warehouse.StockItems
ORDER BY StockItemID;

Commands completed successfully.

CustomerID | PrecProd1
-----------+----------
1          | 25.00    
2          | 25.00    
3          | 25.00    
4          | 25.00    
5          | 25.00    
6          | 25.00    
7          | 25.00    
8          | 25.00    
9          | 25.00    
10         | 25.00    
11         | 25.00    
12         | 25.00    
13         | 25.00    
14         | 25.00    
15         | 25.00    
16         | 25.00    
17         | 25.00    
18         | 25.00    
19         | 25.00    
20         | 25.00    
21         | 25.00    
22         | 25.00    
23         | 25.00    
24         | 25.00    
25         | 25.00    
26         | 25.00    
27         | 25.00    
28         | 25.00    
29         | 25.00    
30         | 25.00    
31         | 25.00    
32         | 25.00    
33         | 25.00    
34         | 25.00    
35         | 25.00    
36         | 25.00    
37         | 25.00    
38         | 25.00    
39         | 25.00    
40         | 25.00    


### Función de Tabla

In [ ]:
USE Pampero
GO
CREATE FUNCTION [dbo].[GetClieOrdenes]
(@cid AS char(5)) RETURNS TABLE
AS
RETURN
SELECT IDPedido, IDCliente, IDEmpleado, FechaPedido, FechaRequerida,FechaEnvio, EnvioPor, Flete, NombreEnvio, DireccionEnvio, CiudadEnvio, RegionEnvio, 
CodigoPostalEnvio, PaisEnvio
FROM Pedidos
WHERE IDCliente = @cid;

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.032